# 01 · EDA: RetailRocket events
ดูข้อมูลให้ละเอียดก่อนเขียน ETL เพื่อให้ทุกกฎ cleaning และการแบ่งข้อมูลมีตัวเลขรองรับ

| # | หัวข้อ | ใช้ตัดสินใจเรื่อง |
| --- | --- | --- |
| 1 | คุณภาพข้อมูล | กฎ validate (Pandera) |
| 2 | ช่วงเวลา | วันที่แบ่ง Train / Val / Test |
| 3 | สัดส่วน event | น้ำหนัก event ตอนเทรน ALS |
| 4 | ผู้ใช้และบอท | เกณฑ์ตัดผู้ใช้ผิดปกติ |
| 5 | Session | ขนาด session และ session features |
| 6 | สินค้า (long-tail) | popularity baseline, cold item |
| 7 | Funnel | conversion และ ground truth |
| 8 | Warm / cold | ความยากของโจทย์ใน Val/Test |
| 9 | ไฟล์เสริม | ฟีเจอร์สินค้าและหมวดหมู่ |

In [ ]:
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

pl.Config.set_tbl_rows(20)
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3})

RAW = Path("../data/raw")
events = pl.read_csv(RAW / "events.csv").with_columns(
    pl.from_epoch("timestamp", time_unit="ms").alias("ts")
)
events.head()

## 1. คุณภาพข้อมูล

In [ ]:
print("rows:", f"{events.height:,}")
print("users:", f"{events['visitorid'].n_unique():,}", "| items:", f"{events['itemid'].n_unique():,}")
print("exact duplicate rows:", int(events.is_duplicated().sum()))
print("duplicate (ts, user, item, event):", events.height - events.unique(["timestamp", "visitorid", "itemid", "event"]).height)
print("\nnulls per column:")
events.null_count()

In [ ]:
# transactionid ควรมีค่าเฉพาะ event = transaction
events.group_by("event").agg(
    pl.len().alias("rows"),
    pl.col("transactionid").is_not_null().sum().alias("has_transactionid"),
)

## 2. ช่วงเวลาและวันที่แบ่งข้อมูล

In [ ]:
t_min, t_max = events["ts"].min(), events["ts"].max()
print("range:", t_min, "->", t_max, f"({(t_max - t_min).days} days)")

daily = events.group_by(pl.col("ts").dt.date().alias("day")).agg(pl.len().alias("n")).sort("day")
plt.plot(daily["day"], daily["n"])
plt.title("Events per day"); plt.show()

In [ ]:
# ตามแผน: Test = 7 วันสุดท้าย, Val = 7 วันก่อนหน้า, Train = ที่เหลือ
test_start = t_max - timedelta(days=7)
val_start = test_start - timedelta(days=7)
print("val_start :", val_start)
print("test_start:", test_start)

split = events.with_columns(
    pl.when(pl.col("ts") >= test_start).then(pl.lit("test"))
    .when(pl.col("ts") >= val_start).then(pl.lit("val"))
    .otherwise(pl.lit("train")).alias("split")
)
split.group_by("split").agg(
    pl.len().alias("events"),
    pl.col("visitorid").n_unique().alias("users"),
    pl.col("ts").min().alias("from"), pl.col("ts").max().alias("to"),
).sort("from")

## 3. สัดส่วน event

In [ ]:
mix = events.group_by("event").len().with_columns(
    (pl.col("len") / events.height * 100).round(2).alias("pct")
).sort("len", descending=True)
display(mix)
plt.bar(mix["event"], mix["len"]); plt.yscale("log"); plt.title("Events by type (log scale)"); plt.show()

In [ ]:
# สัดส่วน event รายสัปดาห์ ว่านิ่งหรือเปลี่ยนตามเวลา (drift)
weekly = (events.group_by(pl.col("ts").dt.truncate("1w").alias("week"), "event").len()
          .pivot(on="event", index="week", values="len").sort("week").fill_null(0))
for e in ["view", "addtocart", "transaction"]:
    plt.plot(weekly["week"], weekly[e] / weekly.select(pl.sum_horizontal(pl.exclude("week")))[:, 0], label=e)
plt.yscale("log"); plt.legend(); plt.title("Weekly event share (log)"); plt.show()

## 4. ผู้ใช้และบอท

In [ ]:
per_user = events.group_by("visitorid").agg(
    pl.len().alias("n"),
    (pl.col("event") == "view").sum().alias("views"),
    (pl.col("event") == "addtocart").sum().alias("carts"),
    (pl.col("event") == "transaction").sum().alias("buys"),
    pl.col("itemid").n_unique().alias("items"),
    pl.col("ts").min().alias("first"), pl.col("ts").max().alias("last"),
)
qs = [0.5, 0.9, 0.99, 0.999, 0.9999, 1.0]
pl.DataFrame({"quantile": qs, "events_per_user": [per_user["n"].quantile(q) for q in qs]})

In [ ]:
print(f"users with exactly 1 event: {(per_user['n'] == 1).mean():.1%}")
print(f"users with >= 2 events : {(per_user['n'] >= 2).mean():.1%}")
print(f"users with >= 5 events : {(per_user['n'] >= 5).mean():.1%}")

plt.hist(per_user.filter(pl.col("n") <= 50)["n"], bins=50)
plt.yscale("log"); plt.title("Events per user (<= 50, log y)"); plt.show()

In [ ]:
# ผู้ใช้ที่ถือ event เป็นสัดส่วนมากผิดปกติ
sorted_n = per_user.sort("n", descending=True)
total = events.height
rows = []
for q in [0.99, 0.999, 0.9999]:
    thr = per_user["n"].quantile(q)
    top = per_user.filter(pl.col("n") > thr)
    rows.append({"cutoff": f"p{q*100:g}", "threshold": thr, "users_removed": top.height,
                 "events_removed": top["n"].sum(), "pct_events": round(top["n"].sum() / total * 100, 2)})
pl.DataFrame(rows)

In [ ]:
sorted_n.head(15)

In [ ]:
# ลักษณะบอท: ดูมาก ไม่ซื้อ เปิดสินค้าหลากหลายผิดปกติ
p999 = per_user["n"].quantile(0.999)
heavy = per_user.filter(pl.col("n") > p999).with_columns(
    ((pl.col("last") - pl.col("first")).dt.total_hours() + 1).alias("hours_active"),
).with_columns((pl.col("n") / pl.col("hours_active")).alias("events_per_hour"))
print("heavy users:", heavy.height, "| with any purchase:", (heavy["buys"] > 0).sum())
heavy.select(["n", "views", "carts", "buys", "items", "hours_active", "events_per_hour"]).describe()

In [ ]:
# เทียบอัตรา buy ของ heavy users กับผู้ใช้ทั่วไป
normal = per_user.filter(pl.col("n") <= p999)
for name, d in [("normal", normal), ("heavy", heavy)]:
    print(f"{name:7s} users={d.height:>9,} buy_rate={d['buys'].sum()/d['n'].sum():.4%} cart_rate={d['carts'].sum()/d['n'].sum():.4%}")

## 5. Session
นิยาม session = event ของผู้ใช้คนเดียวกันที่ห่างกันไม่เกิน 30 นาที

In [ ]:
GAP = pl.duration(minutes=30)
ev = (events.sort(["visitorid", "ts"])
      .with_columns(pl.col("ts").diff().over("visitorid").alias("gap"))
      .with_columns((pl.col("gap").is_null() | (pl.col("gap") > GAP)).cast(pl.Int32).alias("new"))
      .with_columns(pl.col("new").cum_sum().over("visitorid").alias("session")))
sessions = ev.group_by(["visitorid", "session"]).agg(
    pl.len().alias("n"),
    (pl.col("ts").max() - pl.col("ts").min()).dt.total_seconds().alias("dur_s"),
    (pl.col("event") == "addtocart").any().alias("cart"),
    (pl.col("event") == "transaction").any().alias("buy"),
)
print("sessions:", f"{sessions.height:,}", "| per user (mean):", round(sessions.height / per_user.height, 2))
qs = [0.5, 0.75, 0.9, 0.99, 0.999]
pl.DataFrame({"quantile": qs,
              "events_per_session": [sessions["n"].quantile(q) for q in qs],
              "duration_min": [round(sessions["dur_s"].quantile(q) / 60, 1) for q in qs]})

In [ ]:
print(f"single-event sessions: {(sessions['n'] == 1).mean():.1%}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].hist(sessions.filter(pl.col("n") <= 30)["n"], bins=30); ax[0].set_yscale("log"); ax[0].set_title("Events per session (<= 30)")
ax[1].hist(sessions.filter((pl.col("dur_s") > 0) & (pl.col("dur_s") <= 3600))["dur_s"] / 60, bins=60)
ax[1].set_yscale("log"); ax[1].set_title("Session duration, min (0-60)"); plt.show()

In [ ]:
# session ที่ยาวขึ้น มีโอกาส cart/buy มากขึ้นไหม (บอกว่าควรใช้ session features หรือไม่)
(sessions.with_columns(pl.col("n").clip(upper_bound=10).alias("len_bucket"))
 .group_by("len_bucket").agg(pl.len().alias("sessions"),
                              pl.col("cart").mean().round(4).alias("p_cart"),
                              pl.col("buy").mean().round(4).alias("p_buy"))
 .sort("len_bucket"))

## 6. สินค้า (long-tail)

In [ ]:
per_item = events.group_by("itemid").agg(
    pl.len().alias("n"),
    (pl.col("event") == "transaction").sum().alias("buys"),
    pl.col("visitorid").n_unique().alias("users"),
).sort("n", descending=True)
print("items:", f"{per_item.height:,}", "| median events/item:", per_item["n"].median())
for pct in [0.01, 0.05, 0.2]:
    k = int(per_item.height * pct)
    print(f"top {pct:.0%} items ({k:,}) = {per_item.head(k)['n'].sum()/events.height:.1%} of events")
print(f"items with <= 5 events: {(per_item['n'] <= 5).mean():.1%}")

In [ ]:
cum = per_item["n"].cum_sum() / events.height
plt.plot([i / per_item.height for i in range(per_item.height)], cum)
plt.xlabel("fraction of items (most popular first)"); plt.ylabel("cumulative share of events")
plt.title("Item popularity concentration"); plt.show()

In [ ]:
per_item.head(10)

In [ ]:
# ต้อง ALS ต้องมีสินค้า/ผู้ใช้ขั้นต่ำกี่ event ถึงจะเหลือข้อมูลพอ
rows = []
for m in [1, 2, 3, 5]:
    u = per_user.filter(pl.col("n") >= m)["visitorid"]
    i = per_item.filter(pl.col("n") >= m)["itemid"]
    kept = events.filter(pl.col("visitorid").is_in(u) & pl.col("itemid").is_in(i))
    rows.append({"min_events": m, "users": u.len(), "items": i.len(), "events_kept": kept.height,
                 "pct_events": round(kept.height / events.height * 100, 1)})
pl.DataFrame(rows)

## 7. Funnel และ ground truth

In [ ]:
u_view = per_user.filter(pl.col("views") > 0).height
u_cart = per_user.filter(pl.col("carts") > 0).height
u_buy = per_user.filter(pl.col("buys") > 0).height
print(f"users view -> cart -> buy: {u_view:,} -> {u_cart:,} ({u_cart/u_view:.1%}) -> {u_buy:,} ({u_buy/u_view:.1%})")
print(f"buyers that also have addtocart: {per_user.filter((pl.col('buys')>0)&(pl.col('carts')>0)).height/u_buy:.1%}")
print(f"unique transactions: {events['transactionid'].n_unique():,}")

In [ ]:
# ผู้ใช้ที่มี ground truth (addtocart หรือ transaction) ในแต่ละช่วง
gt = split.filter(pl.col("event") != "view")
gt.group_by("split").agg(pl.col("visitorid").n_unique().alias("users_with_gt"),
                         pl.len().alias("gt_events")).sort("split")

## 8. Warm vs cold (ความยากของโจทย์)

In [ ]:
train_users = set(split.filter(pl.col("split") == "train")["visitorid"].unique().to_list())
train_items = set(split.filter(pl.col("split") == "train")["itemid"].unique().to_list())
rows = []
for s in ["val", "test"]:
    g = gt.filter(pl.col("split") == s)
    users = g["visitorid"].unique()
    items = g["itemid"].unique()
    rows.append({"split": s, "gt_users": users.len(),
                 "warm_users_pct": round(sum(u in train_users for u in users.to_list()) / users.len() * 100, 1),
                 "gt_items": items.len(),
                 "seen_items_pct": round(sum(i in train_items for i in items.to_list()) / items.len() * 100, 1)})
pl.DataFrame(rows)

## 9. ไฟล์เสริม

In [ ]:
cat = pl.read_csv(RAW / "category_tree.csv")
print("categories:", cat.height, "| roots:", cat["parentid"].is_null().sum())
cat.head()

In [ ]:
props = pl.scan_csv(RAW / "item_properties_part1.csv", infer_schema=False)
top_props = props.group_by("property").len().sort("len", descending=True).head(12).collect()
top_props

In [ ]:
# property ที่ตั้งใจใช้: categoryid และ available
for p in ["categoryid", "available"]:
    sub = props.filter(pl.col("property") == p).select(
        pl.len().alias("rows"), pl.col("itemid").n_unique().alias("items"), pl.col("value").n_unique().alias("distinct_values")
    ).collect()
    print(p, sub.to_dicts())